<a href="https://colab.research.google.com/github/tevfikaytekin/CMP2003/blob/main/array_based_lists.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Array-Based Lists in C++

## 1. Introduction to Array-Based Lists
An **Array-Based List** is a linear data structure that stores elements in contiguous memory locations. It essentially provides a more flexible way to work with standard arrays by managing the size and capacity automatically.

### Advantages
* **Random Access**: Accessing any element by index is $O(1)$.
* **Cache Locality**: Elements are adjacent in memory, making iteration very fast due to CPU caching.
* **Memory Efficiency**: Minimal overhead compared to linked lists (no pointers per node).

### Disadvantages
* **Fixed Capacity**: Standard arrays cannot grow; we must implement resizing logic.
* **Shifting Overhead**: Inserting or deleting at the beginning or middle requires shifting elements, leading to $O(n)$ performance.

## 2. C++ Class Design
To build this in C++, we use a class to encapsulate the raw pointer to our data, the current number of elements (`size`), and the total allocated space (`capacity`).

In [ ]:
%%writefile array_list.cpp
#include <iostream>
#include <stdexcept>

class ArrayList {
private:
    int* data;      // Pointer to the dynamically allocated array
    int capacity;   // Total space allocated
    int size;       // Current number of elements

    void resize(int newCapacity) {
        capacity = newCapacity;
        int* newData = new int[capacity];
        for (int i = 0; i < size; ++i) {
            newData[i] = data[i];
        }
        delete[] data;
        data = newData;
    }

public:
    ArrayList(int initialCapacity = 2) : capacity(initialCapacity), size(0) {
        data = new int[capacity];
    }

    ~ArrayList() {
        delete[] data;
    }

    void push_back(int value) {
        if (size == capacity) resize(capacity * 2);
        data[size++] = value;
    }

    void insert(int index, int value) {
        if (index < 0 || index > size) throw std::out_of_range("Index out of bounds");
        if (size == capacity) resize(capacity * 2);
        for (int i = size; i > index; --i) {
            data[i] = data[i - 1];
        }
        data[index] = value;
        size++;
    }

    void remove(int index) {
        if (index < 0 || index >= size) throw std::out_of_range("Index out of bounds");
        for (int i = index; i < size - 1; ++i) {
            data[i] = data[i + 1];
        }
        size--;
    }

    int get(int index) const {
        if (index < 0 || index >= size) throw std::out_of_range("Index out of bounds");
        return data[index];
    }

    int find(int value) const {
        for (int i = 0; i < size; ++i) {
            if (data[i] == value) return i;
        }
        return -1;
    }

    void print() const {
        std::cout << "List: ";
        for (int i = 0; i < size; ++i) std::cout << data[i] << " ";
        std::cout << "(Size: " << size << ", Capacity: " << capacity << ")" << std::endl;
    }
};

int main() {
    ArrayList myList(2);
    myList.push_back(10);
    myList.print();
    myList.push_back(20);
    myList.push_back(30); // Triggers resize
    myList.print();

    myList.insert(1, 15);
    myList.print();

    myList.remove(2); // Removes 20
    myList.print();

    std::cout << "Index of 30: " << myList.find(30) << std::endl;
    return 0;
}


Writing array_list.cpp


## 3. Dynamic Resizing and Core Operations
When the list is full, we perform a **Resize**:
1. Allocate a new array (usually double the size).
2. Copy old elements to the new array.
3. Delete the old array memory.

## 4. Time Complexity Analysis
| Function | Time Complexity |
| :--- | :--- |
| **get()** | $O(1)$ |
| **find()** | $O(n)$ |
| **push_back()** | $O(1)$ Amortized |
| **insert()** | $O(n)$ |
| **remove()** | $O(n)$ |

### Understanding Amortized $O(1)$

**Amortized analysis** is a method used to analyze the time complexity of a sequence of operations rather than looking at a single operation in isolation. It calculates the average cost per operation over a worst-case sequence. This ensures that even if one operation is very expensive (like copying an entire array when it resizes), the cost is spread out or "amortized" over many cheap operations, resulting in a low average cost overall.

We use "Amortized $O(1)$" because the expensive $O(n)$ resize operation happens very infrequently. When we double the capacity, we "pay" for the next $n$ insertions in advance. If you spread the cost of one resize across all the insertions that triggered it, the average cost per insertion remains a small constant.

**Concrete Example Calculation:**
Imagine an array with a capacity of 4 that is currently full. We call `push_back()` to add a 5th element.
1. We must allocate a new array of capacity 8.
2. We copy the 4 existing elements (cost = 4 operations).
3. We insert the new 5th element (cost = 1 operation).
*Total cost for this single insertion = 5 operations.*

However, the next 3 `push_back()` calls will just take 1 operation each (since the capacity is 8, no resizing is needed).
* Total cost for adding these 4 new elements = 5 + 1 + 1 + 1 = 8 operations.
* Average cost per insertion = 8 / 4 = **2 operations**.

Even as the array grows to millions of elements, the average cost per insertion will mathematically never exceed a small constant number of operations. This constant upper bound is exactly why we call it $O(1)$ amortized!

## 5. Example Usage and Testing
Finally, we compile and run a test to demonstrate the dynamic behavior.

In [ ]:
!g++ -Wall array_list.cpp -o array_list && ./array_list

List: 10 (Size: 1, Capacity: 2)
List: 10 20 30 (Size: 3, Capacity: 4)
List: 10 15 20 30 (Size: 4, Capacity: 4)
List: 10 15 30 (Size: 3, Capacity: 4)
Index of 30: 2


## Summary
We have successfully implemented a dynamic array-based list in C++. Key takeaways include the importance of constructors/destructors for memory safety and the trade-off between $O(1)$ access and $O(n)$ shifting during modifications.

# Part II: Practice and Assessment

This section combines coding implementation challenges and conceptual questions to test your understanding of Array-Based Lists.

### 6.1 Coding Exercises

Try to implement the following methods for our `ArrayList` class. We recommend tackling them in this order, as some methods will build upon previous ones!

1. **`push_front(int value)`**: Insert an element at the very beginning of the list ($O(n)$).
2. **`shrink_to_fit()`**: If the `size` falls to or below 1/4 of the `capacity`, resize the array to half its current capacity to save memory.
3. **`pop_back()`**: Remove the last element of the list ($O(1)$). *Hint: Utilize `shrink_to_fit()`!*
4. **`clear()`**: Remove all elements from the list, resetting size to 0. *Hint: Utilize `shrink_to_fit()`!*
5. **`removeAll(int value)`**: Find and remove all occurrences of a specific integer. *Hint: Utilize `shrink_to_fit()`!*
6. **`reverse()`**: Reverse the order of elements in the list **in-place** (without creating a second array).
7. **`rotate(int k)`**: Rotate the array elements to the right by `k` positions. **Constraint**: Do this in $O(n)$ time and $O(1)$ space (Hint: you can build this using your `reverse()` logic!). Example: `[1, 2, 3, 4, 5]` rotated by `k=2` becomes `[4, 5, 1, 2, 3]`.

### 6.2 Conceptual Questions

1. **Resizing Strategy**: In our implementation, we double the capacity (`capacity *= 2`) when the list is full. What would be the impact on time complexity if we instead only increased the capacity by a fixed amount (e.g., `capacity += 100`) each time it became full?

2. **Shrink-to-Fit (Hysteresis)**: In the `shrink_to_fit()` exercise, we wait until the list size falls to 1/4 of the capacity before shrinking the capacity to 1/2. Why not shrink it immediately when the size becomes exactly 1/2 of the capacity?

## 7. Solutions

Below are the implementations for each of the exercise methods. Add these to your `ArrayList` class!

### Solution to Exercise 1: `push_front(int value)`
Add this method to insert an element at the very beginning of the list:
```cpp
void push_front(int value) {
    if (size == capacity) resize(capacity * 2);
    for (int i = size; i > 0; --i) data[i] = data[i-1];
    data[0] = value;
    size++;
}
```

### Solution to Exercise 2: `shrink_to_fit()`
Add this method to halve the capacity when the list is underutilized:
```cpp
void shrink_to_fit() {
    if (size > 0 && size <= capacity / 4) {
        std::cout << "[Shrinking to " << capacity / 2 << "] ";
        resize(capacity / 2);
    }
}
```

### Solution to Exercise 3: `pop_back()`
Add this method to remove the last element:
```cpp
void pop_back() {
    if (size > 0) {
        size--;
        shrink_to_fit();
    }
}
```

### Solution to Exercise 4: `clear()`
Add this method to clear the array list:
```cpp
void clear() {
    size = 0;
    shrink_to_fit();
}
```

### Solution to Exercise 5: `removeAll(int value)`
Add this method to efficiently remove all occurrences of a specific integer:
```cpp
void removeAll(int value) {
    int writeIdx = 0;
    for (int readIdx = 0; readIdx < size; ++readIdx) {
        if (data[readIdx] != value) {
            data[writeIdx++] = data[readIdx];
        }
    }
    size = writeIdx;
    shrink_to_fit();
}
```

### Solution to Exercise 6: `reverse()`
Add this method to reverse the elements in-place:
```cpp
void reverse() {
    int start = 0;
    int end = size - 1;
    while (start < end) {
        std::swap(data[start++], data[end--]);
    }
}
```

### Solution to Exercise 7: `rotate(int k)`
Add this method to rotate the array right by `k` positions using the triple-reverse algorithm:
```cpp
void rotate(int k) {
    if (size <= 1) return;
    k = k % size;
    if (k < 0) k += size; // Handle negative rotation

    auto reverseRange = [&](int start, int end) {
        while (start < end) std::swap(data[start++], data[end--]);
    };

    reverseRange(0, size - k - 1);
    reverseRange(size - k, size - 1);
    reverseRange(0, size - 1);
}
```



## 8. Answers to Conceptual Questions

1. **Resizing Strategy**: If we increase capacity by a fixed amount (like +100), the time complexity for $N$ insertions would degrade to $O(N^2)$ overall. Geometric growth (doubling) is essential to maintain the **Amortized $O(1)$** property.

2. **Shrink-to-Fit (Hysteresis)**: Why wait until the list is 1/4 full to shrink to 1/2 capacity? If we shrunk immediately when it was 1/2 full, a series of alternating `push_back` and `pop_back` calls at the threshold would cause the program to resize (copying the whole array) on every single operation, turning $O(1)$ operations into $O(n)$ ones. Hysteresis (the gap between growth and shrink thresholds) preserves amortized efficiency.